In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Demo").master("local[*]").getOrCreate()

# Regression: Predicting Rental Price
In this notebook, we will use the dataset we cleansed in the previous lab to predict Airbnb rental prices

## Load Dataset
Let's load the clean Airbnb dataset in again 
We created it in the previous notebook, it should exists in `/home/jovyan/work/datasets/outpus/airbnb/clean_data`

In [ ]:
file_path = f"/home/jovyan/work/datasets/outpus/airbnb/clean_data"
airbnb_df = spark.read.parquet(file_path)

In [ ]:
train_df, test_df = airbnb_df.randomSplit([.8, .2], seed=42)
print(train_df.cache().count())

In [ ]:
from pyspark.sql.functions import avg, lit, median
from pyspark.ml.evaluation import RegressionEvaluator

avg_price = train_df.select(avg("price")).first()[0]
median_price = train_df.select(median("price")).first()[0]

pred_df = (test_df
          .withColumn("avgPrediction", lit(avg_price))
          .withColumn("medianPrediction", lit(median_price)))

regression_evaluatorAVG = RegressionEvaluator(predictionCol="avgPrediction", labelCol="price", metricName="rmse")
regression_evaluatorMedian = RegressionEvaluator(predictionCol="medianPrediction", labelCol="price", metricName="rmse")


rmseAVG = regression_evaluatorAVG.evaluate(pred_df)
print(f"RMSE for AVG is: {rmseAVG}")
rmseMedian = regression_evaluatorMedian.evaluate(pred_df)
print(f"RMSE for MEDIAN is {rmseMedian}")


r2Avg = regression_evaluatorAVG.setMetricName("r2").evaluate(pred_df)
print(f"R2 for AVG is {r2Avg}")
r2Median = regression_evaluatorMedian.setMetricName("r2").evaluate(pred_df)
print(f"R2 for Median is {r2Median}")

## Linear Regression
Check **`price`** and **`bedrooms`** relations with a visualization

In [ ]:
display(train_df.select(<TODO>))

In [ ]:
display(train_df.select("price", "bedrooms").summary())

Our dataset has a lot of columns, we'll be using only two of them for this notebook for the sake of simplicity
* bedrooms: Feature
* price: Label

We will use [LinearRegression](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.regression.LinearRegression.html?highlight=linearregression#pyspark.ml.regression.LinearRegression) to build the model.
We will also use [VectorAssembler](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.feature.VectorAssembler.html?highlight=vectorassembler#pyspark.ml.feature.VectorAssembler) to build the feature column to the proper type

In [ ]:
#Sample Vector Assembler

from pyspark.ml.linalg import Vectors
from pyspark.ml.feature import VectorAssembler

dataset = spark.createDataFrame(
    [(0, 18, 1.0, 1.0),(1, 22, 3.0, 5.0)],
    ["id", "hour", "mobile", "clicked"])

assembler = VectorAssembler(
    inputCols=["hour", "mobile"],
    outputCol="features")

print("original dataset")
dataset.show(truncate=False)

output = assembler.transform(dataset)

print("result dataset")
output.select("id", "features", "clicked").show(truncate=False)

In [ ]:
from pyspark.ml.feature import VectorAssembler

vec_assembler = VectorAssembler(inputCols=[<TODO>], outputCol=<TODO>)

vec_train_df = vec_assembler.transform(train_df)

In [ ]:
vec_train_df.printSchema()

In [ ]:
from pyspark.ml.regression import LinearRegression

lr = LinearRegression(featuresCol=<TODO>, labelCol=<TODO>)
lr_model = lr.fit(vec_train_df)

In [ ]:
print(lr.explainParams())

## Inspect the Model
We can extract the formula for the lineal regression where:
* Formula: y = Coefficient * X + Intercept

In [ ]:
m = lr_model.coefficients[0]
b = lr_model.intercept

print(f"The formula for the linear regression line is y = {m:.2f}x + {b:.2f}")

## Apply Model to Test Set
* First transform the test_df with Vector Assembler as we did with train_df
* Instead of fit method, which is used to training, we use transform method from the model, it will create a column named prediction

In [ ]:
vec_test_df = <TODO>.transform(<TODO>)

pred_df = <TODO>.transform(<TODO>)

In [ ]:
display(pred_df.select("price", "prediction"))

## Evaluate the Model

In [ ]:
from pyspark.ml.evaluation import RegressionEvaluator

regression_evaluator = RegressionEvaluator(predictionCol=<TODO>, labelCol=<TODO>, metricName=<TODO>)

rmse = regression_evaluator.evaluate(pred_df)
print(f"RMSE is {rmse}")
r2 = regression_evaluator.setMetricName(<TODO>).evaluate(pred_df)
print(f"R2 is {r2}")